# Package

In [1]:
# ----------------------------
# Core
# ----------------------------
from pathlib import Path
import numpy as np
import pandas as pd
from dateutil.relativedelta import relativedelta
import pickle

# ----------------------------
# Feature Store
# ----------------------------
from feast import FeatureStore

# Model
from statsmodels.tsa.ar_model import AutoReg
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Importation des données

In [2]:
# ----------------------------
# Locate Feast repo (notebook-safe)
# ----------------------------
def find_project_root(start: Path, marker: str = "2_data_processing") -> Path:
    p = start.resolve()
    for parent in [p] + list(p.parents):
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(
        f"Impossible de trouver la racine projet (marker '{marker}') depuis {start}"
    )

PROJECT_ROOT = find_project_root(Path.cwd(), marker="2_data_processing")

FEAST_REPO_PATH = (
    PROJECT_ROOT
    / "2_data_processing"
    / "feature_store"
    / "feast_repo"
    / "feature_repo"
)

print("FEAST_REPO_PATH:", FEAST_REPO_PATH)
print("feature_store.yaml exists:", (FEAST_REPO_PATH / "feature_store.yaml").exists())


# ----------------------------
# Load features from Feast
# ----------------------------
def load_features_from_feast(entity_df: pd.DataFrame, feature_refs: list[str]) -> pd.DataFrame:
    fs = FeatureStore(repo_path=str(FEAST_REPO_PATH))
    return fs.get_historical_features(entity_df=entity_df, features=feature_refs).to_df()


# ----------------------------
# Config data
# ----------------------------
START = "1959-01-01"
END   = "2025-09-01"
FREQ  = "MS"
SERIES_ID = "UNRATE"
FEATURE_REFS = ["stationary_value:value"]  # UNRATE déjà stationnaire

dates = pd.date_range(start=START, end=END, freq=FREQ)
entity_df = pd.DataFrame({"series_id": [SERIES_ID] * len(dates), "date": dates})

ts_raw = load_features_from_feast(entity_df, FEATURE_REFS)

ts = (
    ts_raw
    .rename(columns={"series_id": "unique_id", "date": "ds", "value": "y"})
    .sort_values(["unique_id", "ds"])
    .reset_index(drop=True)
)

FEAST_REPO_PATH: D:\Portofolio Data science\Time Series\Explainable_AI_Forecast_and_explain_the_Unemployment_of_USA\2_data_processing\feature_store\feast_repo\feature_repo
feature_store.yaml exists: True
Using date as the event timestamp. To specify a column explicitly, please name it event_timestamp.


# Préparation des données

## Passage en format WIDE

In [3]:
ts_raw = (
    ts_raw
    .pivot(index="date", columns="series_id", values="value")
    .sort_index()
)

print(ts_raw)

series_id                  UNRATE
date                             
1960-01-01 00:00:00+00:00    -0.8
1960-02-01 00:00:00+00:00    -1.1
1960-03-01 00:00:00+00:00    -0.2
1960-04-01 00:00:00+00:00     0.0
1960-05-01 00:00:00+00:00     0.0
...                           ...
2025-05-01 00:00:00+00:00     0.2
2025-06-01 00:00:00+00:00     0.0
2025-07-01 00:00:00+00:00     0.0
2025-08-01 00:00:00+00:00     0.1
2025-09-01 00:00:00+00:00     0.3

[789 rows x 1 columns]


## Construire la série y

In [4]:
# Vérifie que l’index est bien une date (sinon essaie de le convertir)
y = ts_raw.copy()

if not isinstance(y.index, (pd.DatetimeIndex, pd.PeriodIndex)):
    y.index = pd.to_datetime(y.index, errors="coerce")

# aménager la fréquence mensuelle (début de mois)
y.index = y.index.to_period("M").to_timestamp(how="start")
y = y.sort_index().asfreq("MS").astype(float)

# 🔒 borne la date max (sans dropna)
y = y.loc[:pd.Timestamp("2025-08-01")]

# y (DataFrame) → Series 1D
if isinstance(y, pd.DataFrame):
    y = y.iloc[:, 0]

print(
    f"✅ Série prête : {y.index.min().date()} → {y.index.max().date()} "
    f"| n={len(y)} | freq={y.index.freqstr}"
)

✅ Série prête : 1960-01-01 → 2025-08-01 | n=788 | freq=MS


C:\Users\Mita\AppData\Local\Temp\ipykernel_14248\4267632309.py:8: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  y.index = y.index.to_period("M").to_timestamp(how="start")


# Run_config

In [5]:
# ==========================================
# AR(1) — Pseudo-OOS continu (h=12), p=1 fixe + BAGGING (bootstrap en blocs)
# ==========================================
# ---------- Paramètres ----------
h = 12
min_train_n = 36
trend = "c"
p_fixed = 1

# ---------- Nouveaux paramètres bagging ----------
use_bagging = True      # ← interrupteur ON/OFF
B_boot = 30            # nb de ré-échantillonnages
L_block = 12            # taille de bloc (mois) pour moving-block bootstrap
rng = np.random.default_rng(123)  # seed bootstrap

## Boostrap Utilities

In [6]:
# ---------- Utilitaires bootstrap ----------
def moving_block_bootstrap(arr, L, rng):
    """Concatène des blocs contigus de taille L tirés aléatoirement jusqu'à longueur n."""
    n = len(arr)
    if L <= 0 or L > n:
        raise ValueError("L_block invalide")
    nb = int(np.ceil(n / L))
    starts = rng.integers(0, n - L + 1, size=nb)
    out = np.concatenate([arr[s:s+L] for s in starts])[:n]
    return out

def bagged_h_forecast_AR1(y_tr, h, trend, B, L, rng):
    """
    Prévision à horizon h par bagging (residual moving-block bootstrap) pour AR(1).
    Retourne (yhat_mean, yhat_dist, base_pred)
    """
    base_model = AutoReg(y_tr, lags=1, old_names=False, trend=trend).fit()
    base_fc = base_model.predict(start=len(y_tr), end=len(y_tr) + h - 1)
    base_pred = float(base_fc.iloc[-1])

    resid = base_model.resid.values
    fitted = (y_tr.iloc[-len(resid):].values - resid)  # ŷ_t aligné aux résidus

    boot_preds = []
    for _ in range(B):
        res_b = moving_block_bootstrap(resid, L, rng)   # bootstrap des résidus
        y_b = fitted + res_b                             # série bootstrapée
        m_b = AutoReg(pd.Series(y_b, index=y_tr.index[-len(y_b):]),
                      lags=1, old_names=False, trend=trend).fit()
        fc_b = m_b.predict(start=len(y_tr), end=len(y_tr) + h - 1)
        boot_preds.append(float(fc_b.iloc[-1]))
    return float(np.mean(boot_preds)), np.array(boot_preds), base_pred

In [7]:
# ---------- Sécurisation de la série y ----------
y = pd.Series(y.astype(float).values, index=pd.to_datetime(y.index)).asfreq("MS").dropna()
print(f"y: {y.index.min().date()} → {y.index.max().date()}  (n={len(y)}) | freq={y.index.freqstr}")

y: 1960-01-01 → 2025-08-01  (n=788) | freq=MS


In [8]:
# ---------- Boucle pseudo-OOS continue ----------
rows = []
last_model = None
last_fit_end = None

last_t_end = y.index.max() - relativedelta(months=h)

for t_end in y.index:
    if t_end > last_t_end:
        break

    y_tr = y.loc[:t_end]
    if len(y_tr) < max(min_train_n, p_fixed + 1):
        continue

    # fit AR(1) base (utile pour sauvegarde / comparaison)
    ar1 = AutoReg(y_tr, lags=p_fixed, old_names=False, trend=trend).fit()
    last_model = ar1
    last_fit_end = t_end

    # ----- Prévision à h mois (bagging ou base) -----
    if use_bagging:
        yhat_h, yhat_dist, yhat_h_base = bagged_h_forecast_AR1(
            y_tr=y_tr, h=h, trend=trend, B=B_boot, L=L_block, rng=rng
        )
        yhat_p05 = float(np.percentile(yhat_dist, 5))
        yhat_p95 = float(np.percentile(yhat_dist, 95))
    else:
        fc = ar1.predict(start=len(y_tr), end=len(y_tr) + h - 1)
        yhat_h = float(fc.iloc[-1])
        yhat_h_base = yhat_h
        yhat_p05 = np.nan
        yhat_p95 = np.nan

    t_fore = t_end + relativedelta(months=h)
    if t_fore in y.index:
        rows.append((t_fore, yhat_h, float(y.loc[t_fore]), yhat_p05, yhat_p95, yhat_h_base))

In [9]:
# ---------- DataFrame OOS ----------
if rows:
    df_oos_ar1 = (
        pd.DataFrame(rows, columns=["date", "y_hat", "y_true", "y_hat_p05", "y_hat_p95", "y_hat_base"])
          .set_index("date").sort_index()
    )
else:
    df_oos_ar1 = pd.DataFrame(columns=["y_hat", "y_true", "y_hat_p05", "y_hat_p95", "y_hat_base"])
    df_oos_ar1.index = pd.to_datetime(pd.Index([]))

print(f"\n✅ Pseudo-OOS terminé — n prévisions = {len(df_oos_ar1)}")
print(df_oos_ar1.head(3))


✅ Pseudo-OOS terminé — n prévisions = 741
               y_hat  y_true  y_hat_p05  y_hat_p95  y_hat_base
date                                                          
1963-12-01  0.070473     0.0  -0.149798   0.286016   -0.080890
1964-01-01  0.017682    -0.1  -0.194683   0.225812    0.141077
1964-02-01  0.090722    -0.5  -0.050079   0.265508    0.408114


In [10]:
# ---------- (facultatif) Scores par période ----------
if len(df_oos_ar1):
    df_val  = df_oos_ar1.loc["1983-01-01":"1989-12-31"].copy()
    df_test = df_oos_ar1.loc["1990-01-01":"2025-08-31"].copy()

    if len(df_val):
        mae  = mean_absolute_error(df_val["y_true"], df_val["y_hat"])
        rmse = np.sqrt(mean_squared_error(df_val["y_true"], df_val["y_hat"]))
        r2   = r2_score(df_val["y_true"], df_val["y_hat"]) if len(df_val) > 1 else np.nan
        print(f"\n📊 Validation 83–89 — n={len(df_val)} | MAE={mae:.3f} | RMSE={rmse:.3f} | R²={r2:.3f}")

    if len(df_test):
        mae  = mean_absolute_error(df_test["y_true"], df_test["y_hat"])
        rmse = np.sqrt(mean_squared_error(df_test["y_true"], df_test["y_hat"]))
        r2   = r2_score(df_test["y_true"], df_test["y_hat"]) if len(df_test) > 1 else np.nan
        print(f"📊 Test 90–2025 — n={len(df_test)} | MAE={mae:.3f} | RMSE={rmse:.3f} | R²={r2:.3f}")


📊 Validation 83–89 — n=84 | MAE=0.817 | RMSE=1.234 | R²=-0.949
📊 Test 90–2025 — n=428 | MAE=0.867 | RMSE=1.600 | R²=-0.100


In [11]:
# ---------- Sauvegardes ----------
AR1_LAST_PKL  = "AR1_last_trained_model.pkl"
AR1_LAST_META = "AR1_last_trained_model_meta.csv"
AR1_BUNDLE    = "AR1_h12_oos_bundle.pkl"

In [12]:
# 1) modèle final
if last_model is not None:
    try:
        joblib.dump(last_model, AR1_LAST_PKL)
        print(f"💾 Modèle AR(1) sauvegardé → {AR1_LAST_PKL}")
    except Exception:
        with open(AR1_LAST_PKL, "wb") as f:
            pickle.dump(last_model, f)
        print(f"💾 Modèle AR(1) sauvegardé (pickle) → {AR1_LAST_PKL}")

# 2) bundle des sorties
bundle = {
    "oos_predictions": (
        df_oos_ar1.reset_index()
                  .rename(columns={"y_hat": "y_pred"})
                  .assign(date=lambda d: pd.to_datetime(d["date"]).dt.to_period("M").dt.to_timestamp(how="start"))
    ),
    "params": {
        "model": "AR(1)",
        "trend": trend,
        "horizon": h,
        "lag": 1,
        "min_train_n": min_train_n,
        # ---- nouveaux champs ----
        "use_bagging": bool(use_bagging),
        "B_boot": int(B_boot),
        "L_block": int(L_block)
    },
    "meta": {
        "trained_until": str(last_fit_end.date()) if last_fit_end is not None else None,
        "index_freq": "MS",
        "n_obs_y": int(len(y)),
        "n_forecasts": int(len(df_oos_ar1))
    }
}
with open(AR1_BUNDLE, "wb") as f:
    pickle.dump(bundle, f)
print(f"💾 Bundle AR(1) OOS sauvegardé → {AR1_BUNDLE}")

# 3) méta csv
meta_row = {
    "model": "AR(1)",
    "trend": trend,
    "lag": 1,
    "trained_until": str(last_fit_end.date()) if last_fit_end is not None else None,
    "n_obs_y": int(len(y)),
    "n_forecasts": int(len(df_oos_ar1))
}
pd.DataFrame([meta_row]).to_csv(AR1_LAST_META)
print(f"💾 Méta AR(1) sauvegardée → {AR1_LAST_META}")

💾 Modèle AR(1) sauvegardé (pickle) → AR1_last_trained_model.pkl
💾 Bundle AR(1) OOS sauvegardé → AR1_h12_oos_bundle.pkl
💾 Méta AR(1) sauvegardée → AR1_last_trained_model_meta.csv


# 2. Autoregression en choisissant automatiquement l'ordre de p

In [13]:
# ---------- Paramètres ----------
h = 12
min_train_n = 36          # ≥ 3 ans
trend = "c"               # "c" (constante) ou "n" (sans constante)
p_grid = range(1, 13)     # p ∈ {1,…,12}

cv_update_every_months = 36
cv_anchor = pd.Timestamp("1983-01-01")

# Bagging (comme les auteurs)
use_bagging = True
B_boot = 30               # n_boot ≈ 30
L_block = 12              # blocs de 12 mois (annuels)
rng = np.random.default_rng(123)  # seed bootstrap

In [14]:
# ---------- Utils ----------
def months_since(anchor, t):
    return (t.year - anchor.year) * 12 + (t.month - anchor.month)

def moving_block_bootstrap(arr, L, rng):
    """Concatène des blocs contigus de taille L tirés aléatoirement jusqu'à n."""
    n = len(arr)
    L = max(2, min(int(L), n-1))
    nb = int(np.ceil(n / L))
    starts = rng.integers(0, n - L + 1, size=nb)
    out = np.concatenate([arr[s:s+L] for s in starts])[:n]
    return out

def rolling_mae_for_p(y_series, p, h, min_train, trend):
    """MAE rolling à l'horizon h pour un p donné (sur y_series, en respectant l'ordre temporel)."""
    rows = []
    last_t_end = y_series.index.max() - relativedelta(months=h)
    for t_end in y_series.index:
        if t_end > last_t_end:
            break
        y_tr = y_series.loc[:t_end]
        if len(y_tr) < max(min_train, p + 1):
            continue
        model = AutoReg(y_tr, lags=p, old_names=False, trend=trend).fit()
        fc = model.predict(start=len(y_tr), end=len(y_tr) + h - 1)
        yhat_h = float(fc.iloc[-1])
        t_fore = t_end + relativedelta(months=h)
        if t_fore in y_series.index:
            rows.append((t_fore, yhat_h, float(y_series.loc[t_fore])))
    if not rows:
        return np.inf
    tmp = pd.DataFrame(rows, columns=["date", "y_hat", "y_true"]).set_index("date")
    return float(mean_absolute_error(tmp["y_true"], tmp["y_hat"]))

def select_p_by_cv(y_tr, p_grid, h, min_train, trend):
    """Sélectionne p* minimisant le MAE(h) rolling sur l'échantillon d'entraînement courant."""
    best_p, best_score = None, np.inf
    for p in p_grid:
        score = rolling_mae_for_p(y_tr, p, h, min_train, trend)
        if score < best_score:
            best_score, best_p = score, p
    return int(best_p if best_p is not None else 1)

def bagged_h_forecast_ARp(y_tr, p, h, trend, B, L, rng):
    """
    Prévision à horizon h via bagging (residual moving-block bootstrap) pour AR(p).
    Retourne (yhat_mean, yhat_dist, base_pred).
    """
    base = AutoReg(y_tr, lags=p, old_names=False, trend=trend).fit()
    base_fc = base.predict(start=len(y_tr), end=len(y_tr)+h-1)
    base_pred = float(base_fc.iloc[-1])

    resid = base.resid.values
    fitted = (y_tr.iloc[-len(resid):].values - resid)  # ŷ_t aligné

    preds = []
    for _ in range(B):
        res_b = moving_block_bootstrap(resid, L, rng)
        y_b = fitted + res_b
        m_b = AutoReg(pd.Series(y_b, index=y_tr.index[-len(y_b):]),
                      lags=p, old_names=False, trend=trend).fit()
        fc_b = m_b.predict(start=len(y_tr), end=len(y_tr)+h-1)
        preds.append(float(fc_b.iloc[-1]))
    return float(np.mean(preds)), np.array(preds), base_pred

In [15]:
# ---------- Boucle pseudo-OOS ----------
rows = []
last_model = None
last_fit_end = None
current_p = None

last_t_end = y.index.max() - relativedelta(months=h)

for t_end in y.index:
    if t_end > last_t_end:
        break

    y_tr = y.loc[:t_end]
    if len(y_tr) < min_train_n:
        continue

    # Re-CV à partir de 1983-01 tous les 36 mois
    if t_end >= cv_anchor:
        m = months_since(cv_anchor, t_end)
        need_cv = (m % cv_update_every_months == 0)
    else:
        need_cv = False

    if current_p is None and not need_cv:
        current_p = 1  # valeur initiale avant la première CV

    if need_cv:
        current_p = select_p_by_cv(y_tr, p_grid, h, min_train_n, trend)
        print(f"[CV] {t_end.date()} → p* = {current_p}")

    # Fit de référence (utile pour meta/sauvegarde)
    arp = AutoReg(y_tr, lags=current_p, old_names=False, trend=trend).fit()
    last_model = arp
    last_fit_end = t_end

    # Prévision à h mois
    if use_bagging:
        # (Option) reseed par mois pour reproductibilité run-to-run :
        # rng = np.random.default_rng(int(t_end.strftime("%Y%m")))
        yhat_h, yhat_dist, yhat_base = bagged_h_forecast_ARp(
            y_tr=y_tr, p=current_p, h=h, trend=trend,
            B=B_boot, L=L_block, rng=rng
        )
        yhat_p05 = float(np.percentile(yhat_dist, 5))
        yhat_p95 = float(np.percentile(yhat_dist, 95))
    else:
        fc = arp.predict(start=len(y_tr), end=len(y_tr) + h - 1)
        yhat_h = float(fc.iloc[-1])
        yhat_base = yhat_h
        yhat_p05 = np.nan
        yhat_p95 = np.nan

    t_fore = t_end + relativedelta(months=h)
    if t_fore in y.index:
        rows.append((t_fore, yhat_h, float(y.loc[t_fore]),
                     int(current_p), yhat_p05, yhat_p95, yhat_base))

[CV] 1983-01-01 → p* = 5
[CV] 1986-01-01 → p* = 4
[CV] 1989-01-01 → p* = 4
[CV] 1992-01-01 → p* = 4
[CV] 1995-01-01 → p* = 4
[CV] 1998-01-01 → p* = 4
[CV] 2001-01-01 → p* = 4
[CV] 2004-01-01 → p* = 4
[CV] 2007-01-01 → p* = 4
[CV] 2010-01-01 → p* = 4
[CV] 2013-01-01 → p* = 4
[CV] 2016-01-01 → p* = 4
[CV] 2019-01-01 → p* = 4
[CV] 2022-01-01 → p* = 4


In [16]:
# ---------- Résultats ----------
if rows:
    df_oos_arp = (
        pd.DataFrame(rows, columns=["date","y_hat","y_true","p_used","y_hat_p05","y_hat_p95","y_hat_base"])
          .set_index("date").sort_index()
    )
else:
    df_oos_arp = pd.DataFrame(columns=["y_hat","y_true","p_used","y_hat_p05","y_hat_p95","y_hat_base"])
    df_oos_arp.index = pd.to_datetime(pd.Index([]))

print(f"\n✅ Pseudo-OOS terminé — n prévisions = {len(df_oos_arp)}")
print(df_oos_arp.head(3))

# ---------- Scores par période ----------
if len(df_oos_arp):
    df_val  = df_oos_arp.loc["1983-01-01":"1989-12-31"].copy()
    df_test = df_oos_arp.loc["1990-01-01":"2025-08-31"].copy()

    if len(df_val):
        mae  = mean_absolute_error(df_val["y_true"], df_val["y_hat"])
        rmse = np.sqrt(mean_squared_error(df_val["y_true"], df_val["y_hat"]))
        r2   = r2_score(df_val["y_true"], df_val["y_hat"]) if len(df_val) > 1 else np.nan
        print(f"\n📊 Validation 83–89 — n={len(df_val)} | MAE={mae:.3f} | RMSE={rmse:.3f} | R²={r2:.3f}")

    if len(df_test):
        mae  = mean_absolute_error(df_test["y_true"], df_test["y_hat"])
        rmse = np.sqrt(mean_squared_error(df_test["y_true"], df_test["y_hat"]))
        r2   = r2_score(df_test["y_true"], df_test["y_hat"]) if len(df_test) > 1 else np.nan
        print(f"📊 Test 90–2025 — n={len(df_test)} | MAE={mae:.3f} | RMSE={rmse:.3f} | R²={r2:.3f}")


✅ Pseudo-OOS terminé — n prévisions = 741
               y_hat  y_true  p_used  y_hat_p05  y_hat_p95  y_hat_base
date                                                                  
1963-12-01  0.070473     0.0       1  -0.149798   0.286016   -0.080890
1964-01-01  0.017682    -0.1       1  -0.194683   0.225812    0.141077
1964-02-01  0.090722    -0.5       1  -0.050079   0.265508    0.408114

📊 Validation 83–89 — n=84 | MAE=0.819 | RMSE=1.187 | R²=-0.805
📊 Test 90–2025 — n=428 | MAE=0.865 | RMSE=1.644 | R²=-0.161


In [17]:
# ==========================================
# Sauvegardes — AR(p) bagging (h=12)
# ==========================================
ARP_LAST_PKL  = "ARp_last_trained_model.pkl"
ARP_LAST_META = "ARp_last_trained_model_meta.csv"
ARP_BUNDLE    = "ARp_h12_oos_bundle.pkl"

# 1️⃣ Sauvegarde du modèle final (le dernier AR(p) entraîné)
if last_model is not None:
    try:
        joblib.dump(last_model, ARP_LAST_PKL)
        print(f"💾 Modèle AR(p) sauvegardé → {ARP_LAST_PKL}")
    except Exception:
        with open(ARP_LAST_PKL, "wb") as f:
            pickle.dump(last_model, f)
        print(f"💾 Modèle AR(p) sauvegardé (pickle) → {ARP_LAST_PKL}")

# 2️⃣ Sauvegarde du bundle complet : prévisions + paramètres + métadonnées
bundle = {
    "oos_predictions": (
        df_oos_arp.reset_index()
                  .rename(columns={"y_hat": "y_pred"})
                  .assign(date=lambda d: pd.to_datetime(d["date"]).dt.to_period("M").dt.to_timestamp(how="start"))
    ),
    "params": {
        "model": "AR(p)",
        "trend": trend,
        "horizon": h,
        "p_grid": list(p_grid),
        "min_train_n": min_train_n,
        "cv_update_every_months": cv_update_every_months,
        "cv_anchor": str(cv_anchor.date()),
        # ---- paramètres de bagging ----
        "use_bagging": bool(use_bagging),
        "B_boot": int(B_boot),
        "L_block": int(L_block)
    },
    "meta": {
        "trained_until": str(last_fit_end.date()) if last_fit_end is not None else None,
        "index_freq": "MS",
        "n_obs_y": int(len(y)),
        "n_forecasts": int(len(df_oos_arp)),
        "mean_p_used": float(df_oos_arp["p_used"].mean()) if "p_used" in df_oos_arp else np.nan
    }
}

with open(ARP_BUNDLE, "wb") as f:
    pickle.dump(bundle, f)
print(f"💾 Bundle AR(p) OOS sauvegardé → {ARP_BUNDLE}")

# 3️⃣ Sauvegarde d’un petit résumé méta au format CSV
meta_row = {
    "model": "AR(p)",
    "trend": trend,
    "horizon": h,
    "cv_anchor": str(cv_anchor.date()),
    "cv_update_months": cv_update_every_months,
    "trained_until": str(last_fit_end.date()) if last_fit_end is not None else None,
    "n_obs_y": int(len(y)),
    "n_forecasts": int(len(df_oos_arp)),
    "mean_p_used": float(df_oos_arp["p_used"].mean()) if "p_used" in df_oos_arp else np.nan
}

pd.DataFrame([meta_row]).to_csv(ARP_LAST_META, index=False)
print(f"💾 Méta AR(p) sauvegardée → {ARP_LAST_META}")

💾 Modèle AR(p) sauvegardé (pickle) → ARp_last_trained_model.pkl
💾 Bundle AR(p) OOS sauvegardé → ARp_h12_oos_bundle.pkl
💾 Méta AR(p) sauvegardée → ARp_last_trained_model_meta.csv
